# MACHO import

Author: Konstantin Malanchev

Last run: 2026-11-01

https://macho.nci.org.au/

Imports the per-field parquet files produced by `macho.raw.ipynb`.

In [ ]:
import hats_import
import pyarrow as pa
import pyarrow.parquet
from dask.distributed import Client
from hats.pixel_math.spatial_index import SPATIAL_INDEX_COLUMN
from hats_import import CollectionArguments, VerificationArguments, pipeline, pipeline_with_client
from upath import UPath

hats_import.__version__

In [ ]:
parquet_dir = UPath("/astro/store/shire/hats/raw/macho_parquet")
output_path = UPath("/astro/store/shire/hats/tmp_kostya")

In [ ]:
parquet_files = sorted(parquet_dir.glob("macho_*/*.parquet"))
total_rows = sum(pa.parquet.read_metadata(path).num_rows for path in parquet_files)
len(parquet_files), total_rows

In [ ]:
# Light curve values are low precision and repeat a lot, so they compress much better as
# dictionaries than with the hats default byte_stream_split, sky levels being the exception.
# Same for ra / dec / kv / kr. 28% smaller than the hats defaults.
# https://github.com/astronomy-commons/hats/issues/742
lc_byte_stream_split = {"rnormsky", "bnormsky", "ravesky", "bavesky", "obsid"}
parquet_schema = pa.parquet.ParquetFile(parquet_files[0]).schema
use_dictionary = [
    path
    for path in (parquet_schema.column(index).path for index in range(len(parquet_schema)))
    if path in ("ra", "dec", "kv", "kr")
    or (path.startswith("lc.") and path.split(".")[1] not in lc_byte_stream_split)
]
# Integers that are not in the dictionary list would be written plain
column_encoding = {path: "BYTE_STREAM_SPLIT" for path in ("starid", "seqn", "lc.obsid.list.element")}

In [ ]:
args = (
    CollectionArguments(
        completion_email_address="kmalanch@andrew.cmu.edu",
        output_artifact_name="macho",
        output_path=output_path,
        progress_bar=True,
        simple_progress_bar=True,
        # Compression, page size and page index come from the hats defaults.
        write_table_kwargs={
            "column_encoding": column_encoding,
            "use_dictionary": use_dictionary,
        },
    )
    .catalog(
        output_artifact_name="macho",
        input_file_list=parquet_files,
        file_reader="parquet",
        ra_column="ra",
        dec_column="dec",
        sort_columns="starid",
        expected_total_rows=total_rows,
        # Not availble yet
        # byte_pixel_threshold=256 * 1024 * 1024,
        pixel_threshold=50_000,
        highest_healpix_order=10,
        skymap_alt_orders=[2, 4, 6],
    )
    .add_margin(margin_threshold=10.0, is_default=True)
)

In [ ]:
with Client(n_workers=4, threads_per_worker=1, memory_limit="32GB") as client:
    pipeline_with_client(args, client)

In [ ]:
# Spot-check the encodings: BYTE_STREAM_SPLIT, PLAIN, RLE_DICTIONARY or DELTA_BINARY_PACKED as chosen above.
catalog_file = next((output_path / "macho" / "macho" / "dataset").rglob("*.parquet"))
metadata = pa.parquet.read_metadata(catalog_file)
{
    metadata.row_group(0).column(index).path_in_schema: metadata.row_group(0).column(index).encodings
    for index in range(metadata.num_columns)
    if metadata.row_group(0).column(index).path_in_schema
    in (
        SPATIAL_INDEX_COLUMN,
        "starid",
        "ra",
        "dec",
        "kv",
        "rmagave",
        "lc.rmag.list.element",
        "lc.obsid.list.element",
        "lc.airmass.list.element",
        "lc.kv.list.element",
    )
}

In [ ]:
args = VerificationArguments(
    input_catalog_path=output_path / "macho",
    output_path="./verification/macho",
)
pipeline(args)